## 03_test_fl_flw_5_clients.ipynb

## Particionado federado y generación inicial del dataset experimental

Este notebook constituye la primera aproximación al escenario federado considerado en el trabajo. El objetivo principal es validar la división del conjunto de datos CIFAR-10 en cinco clientes independientes y comprobar la viabilidad de obtener métricas energéticas individuales para cada participante mediante CodeCarbon.

Aunque todavía no se implementa agregación de modelos ni coordinación federada entre clientes, esta fase permite simular el comportamiento básico de un entorno de Aprendizaje Federado mediante la ejecución secuencial de entrenamientos locales sobre subconjuntos independientes del mismo dataset.

Además del entrenamiento, se desarrolla la primera versión del esquema de recogida de datos que servirá como base para el dataset experimental del TFM. Para cada cliente se registran automáticamente parámetros de configuración, métricas energéticas, emisiones operativas y tiempos de ejecución, generando registros estructurados que posteriormente pueden exportarse a formato CSV para su análisis.

Las principales validaciones realizadas en esta fase son:

- División del dataset en cinco particiones equivalentes.
- Entrenamiento local independiente por cliente.
- Captura individualizada de energía consumida, emisiones y duración.
- Diseño inicial del esquema de almacenamiento de resultados.
- Generación automática de registros experimentales exportables a CSV.

Esta etapa constituye el primer paso hacia la construcción del dataset de sostenibilidad utilizado posteriormente para comparar escenarios centralizados y federados y para el cálculo de métricas basadas en SCI.

In [1]:
DATA_DIR = "../data"

DATASET = "CIFAR10"

SCENARIO = "federated"
NUM_CLIENTS = 5

BATCH_SIZE = 64

EPOCHS = 5

LEARNING_RATE = 0.001

RANDOM_SEED = 42


1. Carga del Dataset y DataLoader

In [2]:

import torch

import torch.nn as nn

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import datasets
from torchvision import transforms

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# Partición FL-5
# ==================================================

client_size = len(train_dataset) // NUM_CLIENTS

client_datasets = random_split(
    train_dataset,
    [client_size] * NUM_CLIENTS
)

# ==================================================
# DataLoaders de clientes
# ==================================================

client_loaders = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    for ds in client_datasets
]

# ==================================================
# Test loader global
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [3]:
import torch.nn as nn
import torch.optim as optim

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [4]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

3. Función de pérdida y optimización

In [5]:
import torch.optim as optim
import torch.nn as nn
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

4. Función de entrenamiento

In [6]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

5. Función de evaluación

In [7]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

6. Función principal

Crear cliente Flower

In [8]:
from codecarbon import EmissionsTracker
import flwr as fl

class FlowerClient(fl.client.NumPyClient):

    def __init__(
        self,
        model,
        train_loader,
        test_loader
    ):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader

    def get_parameters(
        self,
        config
    ):
        return [
            val.detach().numpy()
            for _, val in self.model.state_dict().items()
        ]

    def fit(
        self,
        parameters,
        config
    ):

        tracker = EmissionsTracker(
            save_to_file=False
        )

        tracker.start()

        tracker.start_task("training")

        train_model(
            self.model,
            self.train_loader,
            optimizer,
            criterion,
            EPOCHS
        )

        training_data = tracker.stop_task()

        tracker.stop()

        metrics = {
            "energy_kwh":
                training_data.energy_consumed,

            "emissions_kg":
                training_data.emissions,

            "duration_s":
                training_data.duration
        }

        return (
            self.get_parameters(config),
            len(self.train_loader.dataset),
            metrics
        )

    def evaluate(
        self,
        parameters,
        config
    ):
        return (
            0.0,
            len(self.test_loader.dataset),
            {"accuracy": 0.0}
        )

In [9]:
client = FlowerClient(
    model=SimpleCNN(),
    train_loader=client_loaders[0],
    test_loader=test_loader
)

print(client)

In [10]:
client.get_parameters({})

[array([[[[ 0.06682586, -0.17049041, -0.1414873 ],
          [ 0.18396574, -0.06537236,  0.05750046],
          [ 0.05938291,  0.06717531, -0.09072623]],
 
         [[ 0.01205509,  0.01269856,  0.1664805 ],
          [ 0.04226178, -0.11893433,  0.11415026],
          [ 0.09989122,  0.09572285, -0.14369215]],
 
         [[ 0.17984846, -0.04569748, -0.11624414],
          [ 0.04520827, -0.18356632,  0.03321262],
          [ 0.05613079,  0.10895795, -0.16999768]]],
 
 
        [[[ 0.10951729,  0.16335197, -0.0282245 ],
          [-0.11128315, -0.17136502, -0.09329829],
          [ 0.17223285, -0.08159832,  0.09566166]],
 
         [[-0.1789728 ,  0.18942259,  0.0057138 ],
          [-0.04433396, -0.17397518, -0.122579  ],
          [ 0.0870543 , -0.05751202, -0.03086478]],
 
         [[ 0.18805574, -0.18932498, -0.13843012],
          [-0.05847948,  0.07703014,  0.18610865],
          [-0.07709729,  0.13811673,  0.11923732]]],
 
 
        [[[-0.18275748,  0.01899837, -0.0605648 ],
       

In [ ]:
client.fit({}, {})

[codecarbon WARNING @ 11:41:34] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 11:41:34] [setup] RAM Tracking...
[codecarbon INFO @ 11:41:34] [setup] CPU Tracking...
[codecarbon INFO @ 11:41:34] [setup] GPU Tracking...
[codecarbon INFO @ 11:41:34] No GPU found.
[codecarbon WARNING @ 11:41:38] We saw that you have a 12th Gen Intel(R) Core(TM) i7-1255U but we don't know it. Please contact us.
[codecarbon WARNING @ 11:41:38] We will use the default power consumption of 4 W per thread for your 12 CPU, so 48W.
[codecarbon WARNING @ 11:41:38] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist, and are readable, at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 11:41:38] CPU Model on constant consumption mode: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon WARNING @ 11:41:38] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon

Epoch 1/5 - Loss: 2.3052
Epoch 2/5 - Loss: 2.3055
Epoch 3/5 - Loss: 2.3053
Epoch 4/5 - Loss: 2.3055


[codecarbon INFO @ 11:42:08] Energy consumed for RAM : 0.000084 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 11:42:08] Delta energy consumed for CPU with cpu_load : 0.000080 kWh, power : 9.623299527793549 W
[codecarbon INFO @ 11:42:08] Energy consumed for All CPU : 0.000080 kWh
[codecarbon INFO @ 11:42:08] 0.000164 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 2.3055


([array([[[[ 0.06682586, -0.17049041, -0.1414873 ],
           [ 0.18396574, -0.06537236,  0.05750046],
           [ 0.05938291,  0.06717531, -0.09072623]],
  
          [[ 0.01205509,  0.01269856,  0.1664805 ],
           [ 0.04226178, -0.11893433,  0.11415026],
           [ 0.09989122,  0.09572285, -0.14369215]],
  
          [[ 0.17984846, -0.04569748, -0.11624414],
           [ 0.04520827, -0.18356632,  0.03321262],
           [ 0.05613079,  0.10895795, -0.16999768]]],
  
  
         [[[ 0.10951729,  0.16335197, -0.0282245 ],
           [-0.11128315, -0.17136502, -0.09329829],
           [ 0.17223285, -0.08159832,  0.09566166]],
  
          [[-0.1789728 ,  0.18942259,  0.0057138 ],
           [-0.04433396, -0.17397518, -0.122579  ],
           [ 0.0870543 , -0.05751202, -0.03086478]],
  
          [[ 0.18805574, -0.18932498, -0.13843012],
           [-0.05847948,  0.07703014,  0.18610865],
           [-0.07709729,  0.13811673,  0.11923732]]],
  
  
         [[[-0.18275748,  0.0189

In [13]:
result = client.fit({}, {})
print(result[2])

[codecarbon WARNING @ 11:43:34] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 11:43:34] >>> Tracker's metadata:
[codecarbon INFO @ 11:43:34]   Platform system: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
[codecarbon INFO @ 11:43:34]   Python version: 3.12.3
[codecarbon INFO @ 11:43:34]   CodeCarbon version: 3.3.1
[codecarbon INFO @ 11:43:34]   Available RAM : 7.632 GB
[codecarbon INFO @ 11:43:34]   CPU count: 12 thread(s) in 1 physical CPU(s)
[codecarbon INFO @ 11:43:34]   CPU model: 12th Gen Intel(R) Core(TM) i7-1255U
[codecarbon INFO @ 11:43:34]   GPU count: 0
[codecarbon INFO @ 11:43:34]   GPU model: 
[codecarbon INFO @ 11:43:34] Energy consumed for RAM : 0.000000 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 11:43:34] Delta energy consumed for CPU with cpu_load : 0.000000 kWh, power : 4.8001458 W
[codecarbon INFO @ 11:43:34] Energy consumed for All CPU : 0.000000 kWh
[codecarbon INFO @ 11:43:34] 0.000000 kWh of electricity an

Epoch 1/5 - Loss: 2.3055
Epoch 2/5 - Loss: 2.3054
Epoch 3/5 - Loss: 2.3055
Epoch 4/5 - Loss: 2.3054


[codecarbon INFO @ 11:44:06] Energy consumed for RAM : 0.000084 kWh. RAM Power : 10.0 W
[codecarbon INFO @ 11:44:06] Delta energy consumed for CPU with cpu_load : 0.000081 kWh, power : 9.65830438327742 W
[codecarbon INFO @ 11:44:06] Energy consumed for All CPU : 0.000081 kWh
[codecarbon INFO @ 11:44:06] 0.000164 kWh of electricity and 0.000000 L of water were used since the beginning.


Epoch 5/5 - Loss: 2.3055
{'energy_kwh': 0.00016433101424770453, 'emissions_kg': 2.8601813029812975e-05, 'duration_s': 27.001098957000067}


✅ Flower llama a fit().

✅ fit() ejecuta entrenamiento real.

✅ CodeCarbon mide durante el entrenamiento.

✅ Las métricas energéticas se devuelven a través de Flower.

✅ Ya no dependemos de CSVs de CodeCarbon.

✅ Tenemos el mecanismo que necesitaremos para SCI.